In [1]:
## 예제 15.1 패키지 설치 및 환경 준비

# 교과서 호환 버전 고정 및 Plotly 이미지 저장 필수 엔진(kaleido 0.2.1) 설치
!pip install "pyautogen[retrievechat]==0.2.35" yfinance plotly==5.24.1 kaleido==0.2.1 chromadb sentence-transformers pypdf -qqq

## 실행 후 런터임 다시 시작

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.1/318.1 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.5/52.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [7]:
## 예제 15.2 OpenAI API 키 설정

import json
import os

# 여기에 본인의 OpenAI API 키를 입력하세요
openai_api_key = os.getenv("OPENAI_API_KEY")

with open('OAI_CONFIG_LIST.json', 'w') as f:
    config_list = [
        { "model": "gpt-4o-mini", "api_key": openai_api_key,},
        { "model": "gpt-4o", "api_key": openai_api_key,},
        { "model": "gpt-image-2", "api_key": openai_api_key,}, # 최신 최상위 이미지 모델
        { "model": "gpt-image-1.5", "api_key": openai_api_key,} # 기존 실습 모델
    ]
    json.dump(config_list, f)

In [2]:
## 예제 15.3 에이전트에 사용할 설정 불러오기

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)  # Colab 시스템 경고 무시

import autogen

config_list = autogen.config_list_from_json(
    "OAI_CONFIG_LIST.json",
    file_location=".",
    filter_dict={
        "model": ["gpt-4o-mini"],  # 토큰 에러를 방지하기 위해 가볍고 강력한 gpt-4o-mini 지정
    },
)

llm_config = {
    "config_list": config_list,
    "temperature": 0,
}

In [5]:
## 예제 15.4 AutoGen의 핵심 구성요소인 UserProxyAgent와 AssistantAgent

from autogen import AssistantAgent, UserProxyAgent

# 어시스턴트 시스템 메시지에 소스 코드를 파일로 저장하라는 절대 규칙 추가
assistant = AssistantAgent(
    name="assistant",
    llm_config=llm_config,
    system_message="""You are a world-class Python developer.
When using yfinance, always handle the Multi-Index by using `.squeeze()` to make the 'Close' column 1-dimensional, or select the specific ticker column explicitly to avoid ValueError.
[CRITICAL RULE] You must always start your code block with a filename comment like `# filename: samsung_stock_price.py` so that the user_proxy will automatically save your code as a physical file on the disk.
Write full code blocks enclosed in ```python and ```. Do NOT say 'TERMINATE' on your own until the user_proxy executes your code and returns the result."""
)

# 유저 프록시는 변경 없음
user_proxy = UserProxyAgent(
    name="user_proxy",
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", "") and x.get("content", "").strip().endswith("TERMINATE"),
    human_input_mode="NEVER",
    max_consecutive_auto_reply=3,
    code_execution_config={"work_dir": "coding", "use_docker": False}
)

[autogen.oai.client: 06-05 08:39:05] {164} WARNING - The API key specified is not a valid OpenAI format; it won't work with the OpenAI-hosted model.


In [ ]:
## 예제 15.5 삼성전자의 3개월 주식 가격 그래프를 그리는 작업 실행

user_proxy.initiate_chat(
    assistant,
    message="""
삼성전자의 지난 3개월 주식 가격 그래프를 그려서 samsung_stock_price.png 파일로 저장해줘.
plotly 라이브러리를 사용하고 그래프 아래를 투명한 녹색으로 채워줘.
값을 잘 확인할 수 있도록 y축은 구간 최소값에서 시작하도록 해줘.
이미지 비율은 보기 좋게 적절히 설정해줘.
"""
)